## ERFD Fake News Detection - Quick Inference Demo
This notebook demonstrates how to automatically download the pre-trained ERFD model and evaluate it on the test set without any manual setup. We extract the core `Classifier` architecture here for transparency.

In [ ]:
# 1. Clean environment, clone repo and install dependencies
%cd /content
!rm -rf ERFD
!git clone https://github.com/HMXHY/ERFD.git
%cd ERFD
!pip install -r require4colab.txt

import sys
import os
sys.path.append(os.getcwd())

In [ ]:
# 2. Download Data and Checkpoints
# Note: If gdown fails due to Google Drive limits, please ensure the file is set to 'Anyone with the link'
!gdown --id 1GF3yC8hKWIDwrxJvZYgTHkzsv8BBhgOH
!unzip -o ERFD_demo_files.zip
print("Data and checkpoints successfully loaded!")

In [ ]:
# 3. Extract Core Model Architecture for Demo (Bypassing training scripts)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# We can safely import these submodules as they don't contain global argparse
from models.bert import RobertaClassifier
from models.fourierattention import FourierAttention
from utils.load_graphdata import load_origindata_test
from result_output.log_result import output_metrics_metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class Classifier(nn.Module):
    def __init__(self, hidden_dim, freq_dim, attn_heads, dropnum):
        super().__init__()
        self.bert = RobertaClassifier()
        self.fourier_attn = FourierAttention(hidden_dim=hidden_dim, attn_heads=attn_heads)
        self.dropout = nn.Dropout(p=dropnum)
        self.out_layer = nn.Sequential(
            nn.Linear(hidden_dim+freq_dim, 256),
            nn.ReLU(),
            nn.Linear(256,2)
        )
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim*2, hidden_dim),
            nn.Sigmoid()
        )
        self.aigc_head = nn.Sequential(
            nn.Dropout(dropnum),
            nn.Linear(hidden_dim+freq_dim, 256),
            nn.ReLU(),
            nn.Linear(256,2)
        )
        self.freq_lowdim = nn.Linear(hidden_dim, freq_dim)
        self.hidden_dim = hidden_dim

    def forward(self, input_ids, attention_masks):
        seq_feat = self.bert(input_ids = input_ids, attention_mask = attention_masks)
        freq_feat = self.fourier_attn(seq_feat.last_hidden_state)
        gate = self.gate(torch.cat([freq_feat, seq_feat[1]], dim=-1))
        weight_freq_feat = self.freq_lowdim(gate * freq_feat)
        gated_output = torch.cat([weight_freq_feat, (1-gate) * seq_feat[1]], dim=-1)
        logit = self.out_layer(self.dropout(gated_output))
        aigc_logits = self.aigc_head(gated_output)
        return logit, aigc_logits

class Testset(Dataset):
    def __init__(self, input_ids, masks, label, max_len):
        self.input_ids = input_ids
        self.masks = masks
        self.label = label
        self.max_len = max_len
    def __getitem__(self, item):
        return {
            'input_ids': self.input_ids[item],
            'attention_mask': self.masks[item],
            'label': torch.tensor(self.label[item], dtype=torch.long),
            'idx': item
        }
    def __len__(self):
        return self.input_ids.size(0)

def create_eval_loader(input_ids, masks, label, max_len, batch_size):
    ds = Testset(input_ids, masks, np.array(label), max_len) 
    return DataLoader(ds, batch_size=batch_size, num_workers=0)

def test_ouput(test_loader, model):
    y_pred, y_test = [], []
    for Batch_data in tqdm(test_loader):
        with torch.no_grad():
            input_ids = Batch_data['input_ids'].to(device)
            attention_mask = Batch_data['attention_mask'].to(device)
            targets = Batch_data['label'].to(device)
            val_out, _ = model(input_ids=input_ids, attention_masks=attention_mask)
            _, val_pred = val_out.max(dim=1)
            y_pred.append(val_pred)
            y_test.append(targets)
    return y_test, y_pred

In [ ]:
# 4. Mock Arguments for Jupyter Environment
class Args:
    dataset_name = 'politifact'
    hidden_dim = 768
    freq_dim = 4
    attn_heads = 1
    dropout_num = 0.4
    max_len = 512
    batch_size = 4
args = Args()

# 5. Load Test Data
print('Loading Test Data...')
test_input_ids, test_masks, test_label = load_origindata_test(args.dataset_name)
test_loader = create_eval_loader(test_input_ids['O'], test_masks['O'], test_label, args.max_len, args.batch_size)

# 6. Load Pre-trained Model
print('Loading Pre-trained Model Parameters...')
model = Classifier(args.hidden_dim, args.freq_dim, args.attn_heads, args.dropout_num).to(device)
model.load_state_dict(torch.load('checkpoints/ERFD/politifact_iter0.m', map_location=device))
model.eval()

# 7. Run Evaluation
print('Running Inference on Test Set...')
y_test, y_pred = test_ouput(test_loader, model)
output_metrics_metrics(y_test, y_pred, 'Origin Test Set')